# Part 2 — Step 7: Train EfficientNet-B0 Classifier

Fine-tunes a pretrained EfficientNet-B0 on ACNE04 patches (binary: acne vs no_acne).

**Why EfficientNet-B0?**
Lightweight (~5M params), strong ImageNet pretraining, fast to fine-tune, and well-suited
for small patch classification tasks.

**Prerequisites:** Run `06_patch_extraction.ipynb` first.

**Output:** `outputs/classifier/best.pth`

In [ ]:
import os
from pathlib import Path

if Path('/content').exists():
    os.chdir('/content/AcneDetection')
print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

PATCH_DIR  = Path('data/patches')
OUT_DIR    = Path('outputs/classifier')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS     = 20
BATCH      = 64
LR         = 1e-4
NUM_WORKERS = 4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Data loaders

Training augmentation bridges the domain gap between ACNE04 face photos and DermNet clinical images:
- **Color jitter** — varies brightness/contrast/saturation to make model robust to different lighting
- **Horizontal flip** — acne is symmetric, lesions appear on either side
- **Random crop** — forces model to focus on local texture, not image position
- **Gaussian blur** — simulates varying camera sharpness

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(PATCH_DIR / 'train', transform=train_tf)
val_ds   = datasets.ImageFolder(PATCH_DIR / 'val',   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_ds.classes  # ['acne', 'no_acne']
print(f'Classes : {class_names}')
print(f'Train   : {len(train_ds)} images')
print(f'Val     : {len(val_ds)} images')

## 2. Model

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Replace classifier head: 1280 → 2 (acne / no_acne)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.classifier[1].in_features, 2),
)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'EfficientNet-B0 ready — {total_params:.1f}M parameters')

## 3. Training loop

In [ ]:
train_losses, val_losses, val_accs = [], [], []
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ── Validate ─────────────────────────────────────────────────────────
    model.eval()
    total_loss = correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            total_loss += criterion(out, labels).item()
            correct += (out.argmax(1) == labels).sum().item()
    avg_val_loss = total_loss / len(val_loader)
    val_acc = correct / len(val_ds)
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
          f'train_loss={avg_train_loss:.4f}  '
          f'val_loss={avg_val_loss:.4f}  '
          f'val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), OUT_DIR / 'best.pth')
        print(f'  checkpoint saved (val_acc={val_acc:.4f})')

torch.save(model.state_dict(), OUT_DIR / 'last.pth')
print(f'\nDone. Best val accuracy: {best_val_acc:.4f}')

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train loss')
axes[0].plot(val_losses,   label='Val loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(val_accs, label='Val accuracy', color='green')
axes[1].axhline(best_val_acc, linestyle='--', color='grey', label=f'Best: {best_val_acc:.4f}')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('EfficientNet-B0 — Training on ACNE04 Patches', fontsize=13)
plt.tight_layout()
OUT_PART2 = Path('outputs/part2')
OUT_PART2.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_PART2 / 'classifier_training_curves.png', dpi=150)
plt.show()
print(f'Saved → outputs/part2/classifier_training_curves.png')